# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

## 1. Unit of analysis + time window


One row represents the daily search performance of one content page. For this assignment I will analyze data from **March 2026**, which is a mid-panel month and avoids using the final outcome period. This allows features to be created from historical observations without introducing future information.

In [4]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727


## 2. Fields: feature / label / context / excluded

## Fields

Features
- impressions
- clicks
- ctr
- avg_position
- visible_queries

Label
- is_declining (derived from later performance)

Context
- client_hash_id
- content_hash_id
- date

Excluded
- trend_pct
- Any field that directly reveals the future outcome.

These fields are excluded because they would introduce data leakage.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
LIMIT 5
""").df()


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification Queries

The following queries verify the unit of analysis, the available data for the selected month, and the completeness of the selected slice.

Query 1 — Verify the grain

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain = con.sql(f"""
SELECT
COUNT(*) AS total_rows,
COUNT(DISTINCT content_hash_id) AS unique_pages,
COUNT(DISTINCT report_date) AS unique_days
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_pages,unique_days
0,9841378,331437,31


Query 2 — Date range

In [10]:
date_range = con.sql(f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

date_range

,start_date,end_date
0,2026-03-01,2026-03-31


Query 3 — Availability (IS TRUE)

In [9]:
availability = con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
AND gsc_data_available IS TRUE
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The dataset is anonymized and contains search performance metrics rather than the reasons behind ranking changes. Client histories are not uniform because some clients have shorter periods of available data. Therefore, this analysis should be interpreted as observed patterns that support decision-making rather than causal explanations or predictions of Google's algorithms.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.